In [1]:
%load_ext autoreload
%autoreload 2

In [184]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sqlalchemy import create_engine
from sqlalchemy.orm import Session
from sqlalchemy.orm import joinedload

import src
import src.data.doccano_models as dm
import src.data.models as m

In [3]:
pd.set_option("display.max_rows", 256)
doccano_engine = create_engine(src.DOCCANO_ENGINE)
ps_engine = create_engine(src.PS_ENGINE)
doccano_session = Session(doccano_engine)
ps_session = Session(ps_engine)

In [4]:
PROJECT = "YT_Pop_1"

In [168]:
doccano_examples = (
    doccano_session.query(dm.ExamplesExample)
    .options(
        joinedload(dm.ExamplesExample.labels),
        joinedload(dm.ExamplesExample.state),
    )
    .join(dm.ExamplesExample.project)
    .join(dm.ExamplesExample.state)
    .filter(
        dm.ProjectsProject.name == PROJECT,
        dm.ExamplesExample.state.any(),
    )
)

In [169]:
def get_video(s, video_id):
    video = s.query(m.Video).filter(m.Video.id == video_id).one()
    return video

In [230]:
rows = []
for ex in doccano_examples:
    row = {}
    try:
        assert len(ex.labels) == 1
    except:  # noqa: E722
        print(ex.text, ex.labels)
        continue

    row["label"] = str(ex.labels[0])
    # if row["label"].startswith("Rede"):
    #     row["label"] = "Rede"

    video_id = ex.meta["id"]
    row["video_id"] = video_id
    row["channel"] = ex.meta["uploader_id"]
    video = get_video(ps_session, video_id)
    row["duration"] = video.duration
    row["title"] = video.title
    row["description"] = video.description
    sents = sorted(video.sentences, key=lambda x: x.sentence_no, reverse=False)
    row["video_intro"] = " ".join(str(tok) for sent in sents[:3] for tok in sent.tokens)
    rows.append(row)


Frank Bsirske – Beitrag zum Jahresauftakt 2019

----------------------------------------

Beratung des Parteivorstandes der LINKEN mit den Vorsitzenden der Bundestagsfraktion und aus den Ländern, u.a. mit Katja Kipping, Bernd Riexinger, Dietmar Bartsch und dem Vorsitzenden der Gewerkschaft ver.di, Frank Bsirske.
 []


In [231]:
df = pd.DataFrame(rows)

le = LabelEncoder()
df.label = df.label.astype(str)
le.fit(df.label)

channel_encoder = LabelEncoder()
df.channel = channel_encoder.fit_transform(df.channel.astype(str))

In [232]:
df.label.value_counts()

label
Rede vor Öffentlichkeit    67
Rede vor Parlament         45
Rede vor Partei            44
Erklärvideo                15
Interview                   6
Podcast                     5
Disskusionsrunde            5
Teaser / Clip               4
Rede vor Demonstration      1
Name: count, dtype: int64

In [235]:
# filter all labels that occur only once
df = df[df.groupby("label").label.transform(len) > 1]

train, test = train_test_split(
    df,
    stratify=df["label"],
    random_state=1337,
    test_size=0.3,
)

In [236]:
train.shape, test.shape

((133, 7), (58, 7))

In [237]:
cols = [
    "title",
    "description",
    "video_intro",
    "channel",
    "duration",
]

X_train = train[cols]
X_test = test[cols]

y_train = le.transform(train["label"])
y_true = le.transform(test["label"])

In [238]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

column_transformer = ColumnTransformer(
    remainder="passthrough",
    transformers=[
        ("title_vectorizer", TfidfVectorizer(use_idf=True, analyzer="char"), "title"),
        ("desc_vectorizer", TfidfVectorizer(use_idf=True, analyzer="char"), "description"),
        ("intro_vectorizer", TfidfVectorizer(use_idf=True, analyzer="char"), "video_intro"),
    ],
)

pipeline = Pipeline(
    [
        ("preprocessing", column_transformer),
        ("scale", StandardScaler()),
        (
            "classifier",
            RandomForestClassifier(
                class_weight="balanced_subsample", n_estimators=500, criterion="entropy",
            ),
        ),
    ],
)

In [239]:
_ = pipeline.fit(X_train, y_train)

In [240]:
y_test = pipeline.predict(X_test)

In [241]:
print(confusion_matrix(y_true, y_test))

[[ 0  0  0  0  0  0  1  0]
 [ 0  0  0  0  1  1  3  0]
 [ 0  0  0  0  0  0  2  0]
 [ 0  0  0  0  0  0  2  0]
 [ 0  0  0  0 13  1  0  0]
 [ 0  0  0  0  0  5  8  0]
 [ 0  0  0  0  3  2 15  0]
 [ 0  0  0  0  0  0  1  0]]


In [242]:
print(classification_report(y_true, y_test, zero_division=0))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.00      0.00      0.00         5
           2       0.00      0.00      0.00         2
           3       0.00      0.00      0.00         2
           5       0.76      0.93      0.84        14
           6       0.56      0.38      0.45        13
           7       0.47      0.75      0.58        20
           8       0.00      0.00      0.00         1

    accuracy                           0.57        58
   macro avg       0.22      0.26      0.23        58
weighted avg       0.47      0.57      0.50        58



In [243]:
le.classes_

array(['Disskusionsrunde', 'Erklärvideo', 'Interview', 'Podcast',
       'Rede vor Demonstration', 'Rede vor Parlament', 'Rede vor Partei',
       'Rede vor Öffentlichkeit', 'Teaser / Clip'], dtype=object)

In [244]:
predicted_labels = le.inverse_transform(y_test)

In [245]:
test["y_test"] = predicted_labels
test.channel = channel_encoder.inverse_transform(test.channel)

In [246]:
test

,label,video_id,channel,duration,title,description,video_intro,y_test
191,Rede vor Öffentlichkeit,MTZoGVjH5Cc,@spdde,95,Ostkonvent 2021. Für Ostdeutschland. Für Dich....,Die SPD steht für Respekt in jeder Hinsicht. W...,Thüringen hat einiges zu bieten . Wunderbare L...,Rede vor Öffentlichkeit
53,Rede vor Öffentlichkeit,7nJ4egPFucM,@AfDFraktionimBundestag,3595,"Anträge der AfD-Fraktion zum ""Globalen Pakt fü...",Folge uns auch auf Telegram: https://t.me/afdf...,"Ja , meine Damen und Herren , lassen Sie uns b...",Rede vor Parlament
15,Rede vor Parlament,IpedFCprDKc,@AfDFraktionimBundestag,272,Die EU kann und darf kein Staat werden! - Fabi...,Folge uns auch auf Telegram: https://t.me/afdf...,"Herr Präsident , meine Damen und Herren , vor ...",Rede vor Parlament
91,Rede vor Öffentlichkeit,0XufSun2-5U,@DieGruenen,221,Gastbeitrag Helga Schmidt | Debatte Internatio...,Helga Schmidt spricht anlässlich der digitalen...,"Danke für die Gelegenheit , eine Brüsseler Sic...",Rede vor Öffentlichkeit
149,Interview,_TQNW0Vq7dc,@csumedia,2834,360-Grad Söder Persönlich Spezial mit Minister...,Exklusiv in 360-Grad ein Söder Persönlich Spez...,"Grüß Gott , liebe Zuschauer , herzlich willkom...",Rede vor Öffentlichkeit
4,Podcast,9G3S08noCt4,@AfDTV,2918,"Nie mehr Abschiebung? | 7 Tage Deutschland, Au...","Antifa-Faeser, die Innenministerin von der SPD...",Guten Tag aus Berlin . Hier ist 7 Tage Deutsch...,Rede vor Öffentlichkeit
11,Rede vor Öffentlichkeit,FNNssNUuCZ8,@DIELINKE,139,Carola Rackete: Ja zu einer anderen Agrarpolit...,Rechte und AfD versuchen für ihre eigenen Zwec...,Zur ursprünglich vom Deutschen Bauernverband o...,Rede vor Öffentlichkeit
76,Rede vor Öffentlichkeit,Ly94UPp-wwk,@DIELINKE,694,Martin Schirdewan: Wahlkampfrede zur Europawah...,#europawahl #linke #dielinke #europasolidarisc...,"Mein Name ist Martin Schürdemann , ich bin mit...",Rede vor Öffentlichkeit
108,Rede vor Öffentlichkeit,2nilhzPDV4E,@FDP,461,Herbstprognose: Stärkung der Wettbewerbsfähigk...,Die deutsche Wirtschaft muss sich in diesem Ja...,Denn die Ursachen für diese Probleme jetzt sin...,Rede vor Öffentlichkeit
92,Rede vor Partei,2TmR3Xt1Bf8,@DieGruenen,327,Omid Nouripour | Politische Rede auf dem Lände...,"#Länderrat2023 Omid Nouripour, Co-Bundesvorsit...","Meine Damen und Herren , auch von mir herzlich...",Rede vor Partei
